# summarize

> Generate section and full summaries from video xscripts via LLM

In [ ]:
#| default_exp summarize

## Design

One LLM call generates all section summaries + a full video summary.
Each section summary includes: summary text, keywords, and evidence quote.

`summaries.json` is **self-contained**: it embeds the relevant `meta.json` and `toc.json` fields so downstream consumers (CLI, learning-map generators, external tools) only need to open one file per video.

**summaries.json format:**
```json
{
  "video": {
    "id": "<VIDEO_ID>", "title": "...", "channel": "...",
    "url": "https://www.youtube.com/watch?v=<VIDEO_ID>",
    "duration": 7007, "upload_date": "20251101"
  },
  "sections": [
    {"path": "1", "title": "Introduction", "start": 0, "end": 137,
     "summary": "...", "keywords": [...],
     "evidence": {"text": "...", "at": 63}}
  ],
  "full": {"summary": "...", "keywords": [...], "evidence": {"text": "...", "at": 123}}
}
```

If `toc.json` is missing, `generate_toc` is called first internally. The LLM call returns `{full, sections: {path: {...}}}` and `_assemble_summaries` merges it with `meta` and `toc_sections` into the canonical shape above.

In [ ]:
#| export
import json
from pathlib import Path
from pydantic import BaseModel, Field

In [ ]:
#| export
from yttoc.core import slice_segments, Segment

def _build_summary_prompt(segments: list[Segment], # Full xscript segments
                          sections: list[dict], # [{path, title, start, end}, ...] from toc.json
                          meta: dict # meta.json content
                         ) -> str: # Prompt for LLM
    "Build prompt asking LLM to summarize each section and the full video."
    parts = []
    for sec in sections:
        sliced = slice_segments(segments, sec['start'], sec['end'])
        lines = []
        for s in sliced:
            mm = int(s.start // 60)
            ss = int(s.start % 60)
            lines.append(f"[{mm:02d}:{ss:02d}] {s.text}")
        parts.append(f"### Section {sec['path']}: {sec['title']}\n" + '\n'.join(lines))

    transcript = '\n\n'.join(parts)
    title = meta.get('title', '')
    channel = meta.get('channel', '')
    desc = meta.get('description', '')

    return f"""You are a structural editor for YouTube video transcripts.

Video info:
- Title: {title}
- Channel: {channel}
- Description: {desc}

Transcript (organized by section):
{transcript}

Task:
For each section AND for the full video, provide:
- summary: 1-2 sentence English summary
- keywords: important terms (people, technical terms, proper nouns)
- evidence: a short quoted phrase from the transcript with its timestamp in seconds

Return a JSON object with:
- "full": {{summary, keywords, evidence: {{text, at}}}}
- "sections": {{"1": {{summary, keywords, evidence: {{text, at}}}}, "2": ...}}"""

In [ ]:
#| export
class Evidence(BaseModel):
    "A quoted phrase from the transcript with its timestamp."
    text: str = Field(description="Short quoted phrase from the transcript")
    at: int = Field(ge=0, description="Timestamp in seconds where the quote appears")

class SectionSummaryPayload(BaseModel):
    "Summary payload for one section or the full video."
    summary: str = Field(description="1-2 sentence English summary")
    keywords: list[str] = Field(description="Important terms (people, technical terms, proper nouns)")
    evidence: Evidence

class SummaryLLMResult(BaseModel):
    "Structured output from the summary generation LLM call."
    full: SectionSummaryPayload
    sections: dict[str, SectionSummaryPayload] = Field(
        description="Per-section summaries keyed by section path ('1', '2', ...)")

def _call_summary_llm(prompt: str) -> dict:
    "Call OpenAI gpt-5.4 with Pydantic-generated schema, return {full, sections} dict."
    import openai
    client = openai.OpenAI()
    response = client.chat.completions.create(
        model='gpt-5.4',
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "generate_summaries",
                "schema": SummaryLLMResult.model_json_schema(),
            },
        },
        messages=[{"role": "user", "content": prompt}],
    )
    return SummaryLLMResult.model_validate_json(
        response.choices[0].message.content).model_dump()

## Tests

In [ ]:
from yttoc.core import Segment
# Test 1: slice_segments returns segments within time range
segs = [
    Segment(start=0, end=5, text='a'),
    Segment(start=5, end=10, text='b'),
    Segment(start=10, end=15, text='c'),
    Segment(start=15, end=20, text='d'),
]
sliced = slice_segments(segs, start=5, end=15)
assert len(sliced) == 2
assert sliced[0].text == 'b'
assert sliced[1].text == 'c'
print('ok')

In [ ]:
# Test 2: slice_segments with no matching segments returns empty
sliced = slice_segments(segs, start=100, end=200)
assert sliced == []
print('ok')

In [ ]:
from yttoc.core import Segment
# Test 3: _build_summary_prompt includes section titles and transcript
segments = [
    Segment(start=0, end=5, text='hello world'),
    Segment(start=5, end=10, text='second part'),
]
sections = [
    {'path': '1', 'title': 'Intro', 'start': 0, 'end': 5},
    {'path': '2', 'title': 'Main', 'start': 5, 'end': 10},
]
meta = {'title': 'Test Video', 'channel': 'Ch'}
prompt = _build_summary_prompt(segments, sections, meta)
assert 'Test Video' in prompt
assert 'Intro' in prompt
assert 'Main' in prompt
assert 'hello world' in prompt
assert 'second part' in prompt
print('ok')

## CLI

In [ ]:
#| export
from fastcore.script import call_parse
from yttoc.core import fmt_duration, format_header, format_toc_line
from yttoc.fetch import _DEFAULT_ROOT, _update_last_used, _glob_srt
from yttoc.xscript import parse_xscript
from yttoc.toc import generate_toc

def _assemble_summaries(meta: dict, # meta.json content
                        toc_sections: list[dict], # [{path, title, start, end}, ...] from toc.json
                        llm_result: dict # {full, sections: {path: {...}}}
                       ) -> dict: # Self-contained summaries.json payload
    "Merge meta + toc + LLM output into the canonical summaries.json shape. Raise if LLM omitted any section."
    missing = [sec['path'] for sec in toc_sections if sec['path'] not in llm_result['sections']]
    if missing:
        raise ValueError(f"LLM omitted summaries for sections: {missing}")
    return {
        'video': {
            'id': meta.get('id'),
            'title': meta.get('title'),
            'channel': meta.get('channel'),
            'url': meta.get('webpage_url'),
            'duration': meta.get('duration'),
            'upload_date': meta.get('upload_date'),
        },
        'sections': [
            {'path': sec['path'], 'title': sec['title'],
             'start': sec['start'], 'end': sec['end'],
             **llm_result['sections'][sec['path']]}
            for sec in toc_sections
        ],
        'full': llm_result['full'],
    }

def _migrate_old_summaries(cached: dict, # Old-format summaries dict {full, sections: {path: {...}}}
                           video_id: str, root: Path
                          ) -> dict: # New-format summaries dict
    "Rebuild a self-contained summaries.json from the legacy {full, sections: {...}} shape."
    meta_path = root / video_id / 'meta.json'
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    toc_sections = generate_toc(video_id, root)  # cached toc.json hit; no LLM call
    return _assemble_summaries(meta, toc_sections, cached)

def generate_summaries(video_id: str, # Exact video_id
                       root: Path = None, # Root cache directory
                       refresh: bool = False, # Delete cached summaries and regenerate
                      ) -> dict: # Self-contained summaries dict
    "Generate summaries.json for a cached video. Returns summaries dict (auto-migrating old shape)."
    root = root or _DEFAULT_ROOT
    d = root / video_id
    meta_path = d / 'meta.json'
    sum_path = d / 'summaries.json'
    srt_files = _glob_srt(d)
    if not (meta_path.exists() and srt_files):
        raise SystemExit(f"Not cached: {video_id}")

    if refresh and sum_path.exists():
        sum_path.unlink()

    if sum_path.exists():
        cached = json.loads(sum_path.read_text(encoding='utf-8'))
        if 'video' in cached:
            return cached
        # Legacy shape: rebuild in place (no LLM call)
        result = _migrate_old_summaries(cached, video_id, root)
        sum_path.write_text(
            json.dumps(result, indent=2, ensure_ascii=False),
            encoding='utf-8')
        return result

    toc_sections = generate_toc(video_id, root)
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    segments = parse_xscript(srt_files[0])
    prompt = _build_summary_prompt(segments, toc_sections, meta)
    llm_result = _call_summary_llm(prompt)
    result = _assemble_summaries(meta, toc_sections, llm_result)

    sum_path.write_text(
        json.dumps(result, indent=2, ensure_ascii=False),
        encoding='utf-8')
    _update_last_used(meta_path)
    return result

def _print_section_summary(s: dict, url: str):
    "Render one section as a TOC-style header followed by summary/keywords/evidence."
    print(f"## {format_toc_line(s, url)}")
    print(s['summary'])
    print(f"**Keywords:** {', '.join(s['keywords'])}")
    print(f"**Evidence:** \"{s['evidence']['text']}\" [{fmt_duration(s['evidence']['at'])}]")

@call_parse
def yttoc_sum(video_id: str, # Exact video_id
              section: str = '', # Section path (e.g. "3"); empty for all
              root: str = None, # Root cache directory
              refresh: bool = False, # Regenerate summaries
             ):
    "Display summaries for a cached video."
    root = Path(root) if root else _DEFAULT_ROOT
    sums = generate_summaries(video_id, root, refresh=refresh)
    video = sums['video']
    url = video.get('url') or ''

    print(format_header(video))
    print()

    if section:
        s = next((sec for sec in sums['sections'] if sec['path'] == section), None)
        if s is None:
            raise SystemExit(f"Section {section} not found")
        _print_section_summary(s, url)
    else:
        for s in sums['sections']:
            _print_section_summary(s, url)
            print()

        print("## Full Summary")
        print(sums['full']['summary'])
        print(f"**Keywords:** {', '.join(sums['full']['keywords'])}")
        print(f"**Evidence:** \"{sums['full']['evidence']['text']}\" [{fmt_duration(sums['full']['evidence']['at'])}]")
        if url: print(url)

In [ ]:
# Test 4: generate_summaries returns cached summaries.json without LLM call
from tempfile import TemporaryDirectory
import io, contextlib

def _make_test_summaries(video_id: str, url: str = ''):
    "Build a self-contained summaries.json fixture with the given id/url."
    return {
        'video': {
            'id': video_id, 'title': 'T', 'channel': 'C',
            'url': url, 'duration': 600, 'upload_date': '20260101',
        },
        'sections': [
            {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300,
             'summary': 'Intro section.', 'keywords': ['intro'],
             'evidence': {'text': 'hi', 'at': 0}},
            {'path': '2', 'title': 'Main', 'start': 300, 'end': 600,
             'summary': 'Main section.', 'keywords': ['main'],
             'evidence': {'text': 'bye', 'at': 300}},
        ],
        'full': {'summary': 'Full video about testing.', 'keywords': ['test'],
                 'evidence': {'text': 'hello', 'at': 0}},
    }

with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID1'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID1', 'title': 'T', 'channel': 'C', 'duration': 600,
        'upload_date': '20260101', 'last_used_at': '2000-01-01T00:00:00'}))
    (v / 'summaries.json').write_text(json.dumps(_make_test_summaries('VID1')))

    result = generate_summaries('VID1', root)
    assert result['full']['summary'] == 'Full video about testing.'
    assert result['video']['id'] == 'VID1'
    assert len(result['sections']) == 2
    assert {s['path'] for s in result['sections']} == {'1', '2'}
print('ok')

In [ ]:
# Test 5: yttoc_sum reads summaries.json only (no toc.json), embedded URL flows through
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID2'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID2', 'title': 'Test Video', 'channel': 'Ch', 'duration': 600,
        'upload_date': '20260101', 'last_used_at': '2000-01-01T00:00:00'}))
    fixture = _make_test_summaries('VID2', url='https://youtube.com/watch?v=VID2')
    fixture['video']['title'] = 'Test Video'
    fixture['video']['channel'] = 'Ch'
    (v / 'summaries.json').write_text(json.dumps(fixture))
    # Intentionally NO toc.json — yttoc_sum must not depend on it.

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_sum('VID2', root=str(root))
    out = buf.getvalue()

    assert '# Test Video' in out
    assert '## 1. Intro 0:00-5:00 (5:00) https://youtube.com/watch?v=VID2&t=0' in out
    assert '## 2. Main 5:00-10:00 (5:00) https://youtube.com/watch?v=VID2&t=300' in out
    assert 'Intro section.' in out
    assert '**Keywords:** intro' in out
    assert '## Full Summary' in out
    assert 'Full video about testing.' in out
    assert 'https://youtube.com/watch?v=VID2' in out  # full-summary URL line
print('ok')

In [ ]:
# Test 6: yttoc_sum --section shows only that section (no URL when summaries lacks it)
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID3'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID3', 'title': 'T', 'channel': 'C', 'duration': 600,
        'upload_date': '20260101', 'last_used_at': '2000-01-01T00:00:00'}))
    (v / 'summaries.json').write_text(json.dumps(_make_test_summaries('VID3')))

    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        yttoc_sum('VID3', '2', root=str(root))  # positional section arg
    out = buf.getvalue()

    assert '## 2. Main 5:00-10:00 (5:00)' in out
    assert '&t=' not in out  # no URL → no deep link
    assert 'Main section.' in out
    assert 'Intro section.' not in out  # only section 2
    assert '## Full Summary' not in out
print('ok')

In [ ]:
# Test 7: legacy summaries.json (no 'video' key) is auto-migrated in place — no LLM call
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID4'; v.mkdir()
    (v / 'captions.en.srt').write_text('1\n00:00:00,000 --> 00:00:01,000\nhi\n')
    (v / 'meta.json').write_text(json.dumps({
        'id': 'VID4', 'title': 'Old', 'channel': 'Ch', 'duration': 600,
        'upload_date': '20260101', 'webpage_url': 'https://youtube.com/watch?v=VID4',
        'last_used_at': '2000-01-01T00:00:00'}))
    (v / 'toc.json').write_text(json.dumps({'sections': [
        {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300},
        {'path': '2', 'title': 'Main', 'start': 300, 'end': 600},
    ]}))
    # Legacy shape: dict-keyed sections, no 'video' block
    legacy = {
        'full': {'summary': 'old full', 'keywords': ['k'], 'evidence': {'text': 'q', 'at': 5}},
        'sections': {
            '1': {'summary': 's1', 'keywords': ['kw1'], 'evidence': {'text': 'e1', 'at': 1}},
            '2': {'summary': 's2', 'keywords': ['kw2'], 'evidence': {'text': 'e2', 'at': 2}},
        },
    }
    sum_path = v / 'summaries.json'
    sum_path.write_text(json.dumps(legacy))

    result = generate_summaries('VID4', root)
    assert 'video' in result
    assert result['video']['id'] == 'VID4'
    assert result['video']['url'] == 'https://youtube.com/watch?v=VID4'
    assert isinstance(result['sections'], list)
    assert len(result['sections']) == 2
    assert result['sections'][0]['title'] == 'Intro'    # from toc
    assert result['sections'][0]['summary'] == 's1'     # from legacy LLM payload
    # On disk has been rewritten in new shape:
    on_disk = json.loads(sum_path.read_text())
    assert 'video' in on_disk
print('ok')

In [ ]:
# Test 8: _assemble_summaries raises if LLM omits any toc section (no silent corruption)
toc = [
    {'path': '1', 'title': 'A', 'start': 0, 'end': 100},
    {'path': '2', 'title': 'B', 'start': 100, 'end': 200},
]
llm_partial = {
    'full': {'summary': 'f', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
    'sections': {
        '1': {'summary': 's1', 'keywords': [], 'evidence': {'text': '', 'at': 0}},
        # '2' is missing
    },
}
try:
    _assemble_summaries({'id': 'X', 'title': 't'}, toc, llm_partial)
    assert False, 'should have raised'
except ValueError as e:
    assert "'2'" in str(e)
print('ok')

## get_summaries

In [ ]:
#| export
def get_summaries(video_id: str, # Exact video_id
                  root: Path = None # Root cache directory (default: ~/.cache/yttoc)
                 ) -> dict: # summaries.json content verbatim, or {"error": "..."}
    "Return summaries.json for a cached video. No transformation — file content returned as-is."
    root = root or _DEFAULT_ROOT
    sum_path = root / video_id / 'summaries.json'
    if not sum_path.exists():
        return {'error': f'summaries.json not found for {video_id}'}
    return json.loads(sum_path.read_text(encoding='utf-8'))

In [ ]:
# Test 9: get_summaries returns summaries.json verbatim
with TemporaryDirectory() as d:
    root = Path(d)
    v = root / 'VID_GS'; v.mkdir()
    fixture = {
        'video': {'id': 'VID_GS', 'title': 'T', 'channel': 'C',
                  'url': '', 'duration': 600, 'upload_date': '20260101'},
        'sections': [
            {'path': '1', 'title': 'Intro', 'start': 0, 'end': 300,
             'summary': 's', 'keywords': ['k'],
             'evidence': {'text': 'e', 'at': 10}}
        ],
        'full': {'summary': 'full', 'keywords': ['fk'],
                 'evidence': {'text': 'fe', 'at': 0}},
    }
    (v / 'summaries.json').write_text(json.dumps(fixture))

    result = get_summaries('VID_GS', root)
    assert result == fixture, 'must return file verbatim'
    assert 'full' in result, 'must include full field'
print('ok')

In [ ]:
# Test 10: get_summaries returns error dict when missing
with TemporaryDirectory() as d:
    result = get_summaries('NONEXIST', Path(d))
    assert 'error' in result
print('ok')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()